# Linear Regression vs. Ridge Regression

This notebook compares ordinary Linear Regression and Ridge Regression for predicting real-estate price per unit area. Both models use the same held-out test observations and the same preprocessing pipeline.

**Target note:** This is price per unit area in the dataset's original units, not total house price. The data are historical and intended for learning, not current valuation advice.

In [ ]:
%pip install ucimlrepo pandas numpy matplotlib seaborn scikit-learn


## 1. Load and inspect the data

We retrieve UCI dataset 477 and assign descriptive names to its six input features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 4775
sns.set_theme(style='whitegrid', context='notebook')

dataset = fetch_ucirepo(id=477)
X = dataset.data.features.copy()
y = dataset.data.targets.iloc[:, 0].copy()
X.columns = ['transaction_date', 'house_age', 'distance_mrt', 'num_convenience_stores', 'latitude', 'longitude']
y.name = 'house_price_unit_area'
df = X.join(y)
print(f'{len(df)} rows | {X.shape[1]} features')
display(df.head())
print('Missing values:', int(df.isna().sum().sum()))

## 2. Explore the target and feature correlations

These plots provide context for the prediction task. Correlation is descriptive and does not imply causation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['house_price_unit_area'], kde=True, ax=axes[0], color='#3973ac')
axes[0].set(title='Target distribution', xlabel='Price per unit area', ylabel='Count')
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='vlag', center=0, annot=True, fmt='.2f', ax=axes[1])
axes[1].set_title('Feature and target correlations')
plt.tight_layout()
plt.show()

## 3. Train both models on an identical split

We hold out 30% of the observations for final evaluation. Median imputation and standardization are included in each pipeline, preventing the test data from influencing preprocessing. Ridge's alpha is fixed at 1.0 for this demonstration; it is not tuned on the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED)

def build_model(estimator):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('regressor', estimator),
    ])

models = {
    'Linear Regression': build_model(LinearRegression()),
    'Ridge Regression (alpha=1)': build_model(Ridge(alpha=1.0)),
}
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
print(f'Train observations: {len(X_train)} | Test observations: {len(X_test)}')

## 4. Compare held-out test metrics

Lower MAE and RMSE indicate smaller prediction errors. Higher R² indicates that the model explains more target variation on this test split. Always consider the metrics together and in context.

In [ ]:
rows = []
for name, pred in predictions.items():
    rows.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
        'R²': r2_score(y_test, pred),
    })
test_results = pd.DataFrame(rows).set_index('Model')
display(test_results.round(3))

# A simple descriptive winner based on this test split's R² (not a universal conclusion).
winner = test_results['R²'].idxmax()
print(f'Higher test-set R² on this split: {winner}')

## 5. Plot predictions and residuals

Points near the diagonal have predictions close to observed values. Residuals are actual minus predicted; patterns around zero can indicate systematic model errors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = {'Linear Regression': '#3973ac', 'Ridge Regression (alpha=1)': '#d17a22'}
combined = np.concatenate([y_test.to_numpy()] + list(predictions.values()))
lo, hi = combined.min(), combined.max()
for name, pred in predictions.items():
    axes[0].scatter(y_test, pred, alpha=0.65, s=35, label=name, color=palette[name])
    axes[1].scatter(pred, y_test.to_numpy() - pred, alpha=0.65, s=35, label=name, color=palette[name])
axes[0].plot([lo, hi], [lo, hi], 'k--', label='Ideal prediction')
axes[0].set(title='Actual vs. predicted', xlabel='Actual price per unit area', ylabel='Predicted price per unit area')
axes[0].legend()
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set(title='Test-set residuals', xlabel='Predicted price per unit area', ylabel='Actual − predicted')
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Compare five-fold cross-validation

Five-fold cross-validation repeats the training/evaluation process using different train/validation folds. Its mean and standard deviation show how sensitive performance is to the particular split. The test set remains separate from this calculation.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='r2')
    cv_rows.append({
        'Model': name,
        'Fold R²': np.round(scores, 3).tolist(),
        'Mean CV R²': scores.mean(),
        'CV standard deviation': scores.std(),
    })
cv_results = pd.DataFrame(cv_rows).set_index('Model')
display(cv_results.round(3))

print('Interpret the test metrics and CV results together; small differences may not be meaningful.')

## Conclusion

Use the metric tables and diagnostic plots above to describe how the models compare on this dataset. Ridge shrinks coefficients and can help when predictors are correlated, while ordinary Linear Regression does not apply that penalty. The better result depends on the data and evaluation; do not assume one model must always win. For a formal report, record the displayed test MAE/RMSE/R² and cross-validation mean ± standard deviation after running all cells.